# Client/server encrypted inference — minimal

The **client** owns the keys (CPU-only, no GPU); the **server** builds its session from the client's key bundle — no secret key, it computes blind. Only bytes cross between them.

The CKKS environment comes from the runner (`NB=client_server bash scripts/run_notebooks.sh`), like every notebook here. The two-process version of this protocol is `scripts/utils/probe_client_server.py`.

In [ ]:
import os, tempfile

import numpy as np

from perseus.nn import EncClient, EncLinear, EncSequential, EncServer

client = EncClient()                       # keygen, CPU-only
bundle = client.save_bundle(os.path.join(tempfile.mkdtemp(dir=os.environ.get("TMPDIR")), "bundle"))

d_pad, d_real = 1024, 768
rng = np.random.default_rng(0)
x_real = rng.standard_normal(d_real) * 0.3
blob_x = client.encrypt(x_real)            # -> server
print(f"request on the wire: {len(blob_x) / 1e6:.1f} MB")

In [ ]:
server = EncServer(bundle)                 # session from the bundle: cannot decrypt

W = rng.standard_normal((d_pad, d_pad)) * 0.5 / np.sqrt(d_real)
W[d_real:, :] = 0.0
W[:, d_real:] = 0.0
model = EncSequential(EncLinear("fc", d_pad, d_pad, weight=W.tolist())).bind(server.inf)

blob_y = server.run(model, blob_x)         # -> client
print(f"computed blind; response on the wire: {len(blob_y) / 1e6:.1f} MB")

In [ ]:
out = client.decrypt(blob_y, d=d_real)
x_pad = np.zeros(d_pad)
x_pad[:d_real] = x_real
ref = (x_pad @ W)[:d_real]
rel = np.linalg.norm(out - ref) / np.linalg.norm(ref)
print(f"client-decrypted vs reference: rel_err={rel:.3e}")
assert rel < 0.05